## Project IDM U1 group SE3 ##

### MCDA ###

Best-worst method (BWM) to determine weights 

In [ ]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd

In [ ]:
def bwm_linear(a_B, a_W):
    n = len(a_B)
    B = np.argmin(a_B)
    W = np.argmin(a_W)
    
    c = np.zeros(n + 1)
    c[-1] = 1
    
    A_ub, b_ub = [], []
    for j in range(n):
        # w_B - a_Bj * w_j - xi <= 0
        row = np.zeros(n + 1); row[B] = 1; row[j] -= a_B[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # -w_B + a_Bj * w_j - xi <= 0
        row = np.zeros(n + 1); row[B] = -1; row[j] += a_B[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # w_j - a_jW * w_W - xi <= 0
        row = np.zeros(n + 1); row[j] = 1; row[W] -= a_W[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # -w_j + a_jW * w_W - xi <= 0
        row = np.zeros(n + 1); row[j] = -1; row[W] += a_W[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
    
    A_eq = [np.concatenate([np.ones(n), [0]])]
    b_eq = [1]
    bounds = [(0, None)] * n + [(0, None)]
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method='highs')
    return res.x[:n], res.x[-1]

# Voorbeeld RWS
a_B = [1, 4, 6, 5, 2, 4, 8, 3, 3, 3]
a_W = [8, 3, 2, 2, 6, 3, 1, 4, 3, 4]
w, xi = bwm_linear(a_B, a_W)
print("Gewichten:", np.round(w, 3))
print("xi*:", np.round(xi, 4))

Gewichten: [0.267 0.074 0.05  0.059 0.149 0.074 0.03  0.099 0.099 0.099]
xi*: 0.0297


In [ ]:
df = pd.read_csv("MCDA_Ratings_Merwedebrug_final.csv", sep=";", skiprows = [1,2])

# display(df.head())
# df.keys()

criteria = df["Stakeholder"]
keys = df.keys()
for i in range(1, len(keys) - 1, 2):

    a_b_i = np.array(df.iloc[:, i])
    a_w_i = np.array(df.iloc[:, i + 1])
    a_b_i = list(a_b_i)
    a_w_i = list(a_w_i)

    to_pop = []

    for j in range(len(a_b_i) - 1):
    
        if np.isnan(a_b_i[j]):
            to_pop.append(j)

    for pop in reversed(to_pop):
        a_b_i.pop(pop)  
        a_w_i.pop(pop)  

    a_b_i = np.array(a_b_i)
    a_w_i = np.array(a_w_i)
    
    w, xi = bwm_linear(a_b_i, a_w_i)
    print(f"Weights for criteria for stakeholder {keys[i]}", np.round(w, 3))
    print("xi*:", np.round(xi, 4))
   
    

Weights for criteria for stakeholder AB_RWS [0.296 0.089 0.04  0.071 0.178 0.089 0.119 0.119]
xi*: 0.0593
Weights for criteria for stakeholder AB_MIWM [0.351 0.263 0.211 0.175]
xi*: 0.7018


ValueError: Invalid input for linprog: A_ub must not contain values inf, nan, or None